### Build and run MHCPrime on example datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntranoslab/MHCPrime/blob/main/notebooks/01_inference_quickstart.ipynb)

#### Setup notebook

In [19]:
# imports
from pathlib import Path
import torch
import pandas as pd
from mhcprime import (
    load_example_dataset,
    build_mhcprime_model,
    load_mhcprime_model,
    load_default_mhc_pseudosequences,
    prepare_input_dataframe,
    predict_dataframe,
    print_data_stats,
    get_default_checkpoint_path
)

In [ ]:
# set up directory structure
import os
import sys
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    os.chdir("/content")

    if not Path("MHCPrime").exists():
        !git clone https://github.com/ntranoslab/MHCPrime.git

    os.chdir("/content/MHCPrime")
    !python -m pip install -e .

    REPO_ROOT = Path.cwd()

else:
    NOTEBOOK_DIR = Path.cwd()

    if NOTEBOOK_DIR.name == "notebooks":
        REPO_ROOT = NOTEBOOK_DIR.parent
    else:
        REPO_ROOT = NOTEBOOK_DIR

    os.chdir(REPO_ROOT)

print("Repo root:", REPO_ROOT)

Repo root: /home/pchati/MHCPrime


In [21]:
# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA H100 80GB HBM3


#### Get test dataframes and process

 - Required columns: `seq`, `allele`.
 - The allele should be formatted without colons or stars, for example, HLA-A*02:01 should be formatted as A0201.
 - If a given allele is not in the pseudosequence dictionary, it will be excluded, however, users can update the pseudosequence dictionary with their alleles, given that the selected positions match the remaining alleles.

In [22]:
# small test data
test_data_small = load_example_dataset("small", index_col=0)
test_data_small_processed = prepare_input_dataframe(test_data_small)
print_data_stats(test_data_small_processed)
display(test_data_small_processed.head(10))

Total: 2,794
Pos: 1,397 | 50.00%
Neg: 1,397 | 50.00%
N alleles: 142


,seq,allele,label,n_flank,c_flank,mhc_a_1,mhc_b_1,mhc_c_1,mhc_a_2,mhc_b_2,mhc_c_2,sa_ma
0,AEDPSQLP$$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
1,FEHGDTNEV$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
2,YEQISYIIP$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
3,KEVEETATA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
4,AEILNGKEISA$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
5,AEHPQSPCV$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
6,AEQDFWRAV$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
7,AEQIKHILA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
8,REAFEAIKA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
9,GFSASSLGG$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA


In [23]:
# large test data
test_data_large = load_example_dataset("large", index_col=0)
test_data_large_processed = prepare_input_dataframe(test_data_large)
print_data_stats(test_data_large_processed)
display(test_data_large_processed.head(10))

Total: 62,074
Pos: 31,037 | 50.00%
Neg: 31,037 | 50.00%
N alleles: 142


,seq,allele,label,n_flank,c_flank,mhc_a_1,mhc_b_1,mhc_c_1,mhc_a_2,mhc_b_2,mhc_c_2,sa_ma
0,AEQIKHILA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
1,MEDETQKEL$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
2,AEIEPEAQA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
3,KESEVLLQA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
4,AEDPSQLP$$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
5,AEFKEAFQL$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
6,KEVEETATA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
7,REYWNEQSA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
8,KEDSQPPTP$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA
9,REVQGFESA$$$$$,B4101,1,$$$$$$$$$$,$$$$$$$$$$,None,YHTKYREISTNTYESNLYWRYNYYTWAVDAYTWY,None,None,None,None,SA


#### Load model and MHCPrime weights

In [24]:
# get mhcprime without weights
model, tokenizer, model_params = build_mhcprime_model(
    device=device,
    print_params=True,
)

model.eval()

print("Model class:", type(model).__name__)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model device:", next(model.parameters()).device)
print("Number of model params:", sum(p.numel() for p in model.parameters()))

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model class: MHCPrime
Tokenizer vocab size: 25
Model device: cuda:0
Number of model params: 25751991


In [25]:
# load trained mhcprime

print("Default checkpoint:", get_default_checkpoint_path())

base_model, tokenizer, model_params = load_mhcprime_model(
    device=device,
    strict=True,
    eval_mode=True,
    print_params=True,
)

print("Model device:", next(base_model.parameters()).device)
print("Training mode:", base_model.training)

Default checkpoint: /home/pchati/MHCPrime/src/mhcprime/checkpoints/mhcprime_base/model_final.pt
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model device: cuda:0
Training mode: False


#### Inference

 - The prediction wrapper accepts both processed and unprocessed dataframes. If entering in a preprocessed dataframe, use `preprocess = False`. The function by default outputs the unprocessed and unpadded dataframe, this can be changed with `return_processed`.
 - There is a slow and fast version of inference that caches the dataframes.
 - The faster approach accepts a cache path where the data is stored. This can optionally be removed after inference.
 - The ranking dataframe is derived from the human proteome. The rank datafrmae is also scored without flanks, as the base model was trained. We rank against a global background; this avoids scoring unseen alleles or pseudosequences when introduced to the model and aligns with how the base model was trained. The ranking dataframe can be altered, however, this requires you to score the inputted rank dataframe prior to ranking.
 - The fast version does perform a dtype conversion. To restore the original dtype to run or train the model again after, please use `restore_model_dtype = True` in the fast version.
 - The raw scores are unbounded, however, we found a reasonable threshold around -100 to -50 for the base model. We suggest using ranks for threshold-based evaluation and raw scores for comparisons against a background/negative distribution.

In [26]:
%%time
# slow inference on smaller dataframe
scored_slow = predict_dataframe(
    model=base_model,
    df=test_data_small_processed,
    tokenizer=tokenizer,
    score_col="mhcprime",
    mode="slow",
    batch_size=1024,
    num_workers=0,
    device=device,
    preprocess=False,
    disable_tqdm=False,
)

display(scored_slow[["seq", "allele", "label", "mhcprime", "mhcprime_rank"]].head(10))

Ranking score columns: 100%|██████████| 1/1 [00:00<00:00, 239.24it/s]


,seq,allele,label,mhcprime,mhcprime_rank
0,AEDPSQLP,B4101,1,7.275182,99.914841
1,FEHGDTNEV,B4101,1,-1.875151,99.666397
2,YEQISYIIP,B4101,1,1.914668,99.818001
3,KEVEETATA,B4101,1,34.041199,99.988327
4,AEILNGKEISA,B4101,1,70.401825,99.998413
5,AEHPQSPCV,B4101,1,20.287800,99.971970
6,AEQDFWRAV,B4101,1,50.382465,99.995392
7,AEQIKHILA,B4101,1,21.302309,99.973770
8,REAFEAIKA,B4101,1,2.652856,99.838631
9,GFSASSLGG,B4101,1,12.615036,99.948708


CPU times: user 14.3 s, sys: 303 ms, total: 14.7 s
Wall time: 1.04 s


In [27]:
%%time
# fast cached inference on larger dataframe
CACHE_PATH = REPO_ROOT / "dataset_cache.pt"

scored_fast = predict_dataframe(
    model=base_model,
    df=test_data_large_processed,
    tokenizer=tokenizer,
    score_col="mhcprime",
    mode="fast",
    batch_size=3072,
    num_workers=8,
    device=device,
    preprocess=False,
    cache_path=CACHE_PATH,
    build_cache=True,
    remove_cache=True,
    use_bf16=True,
    restore_model_dtype=True,
    compile_model=False,
    disable_tqdm=False,
)

display(scored_fast[["seq", "allele", "label", "mhcprime", "mhcprime_rank"]].head(10))
print("Cache exists after inference:", CACHE_PATH.exists())

Building token cache:   0%|          | 0/62074 [00:00<?, ?it/s]

Building token cache: 100%|██████████| 62074/62074 [00:02<00:00, 23594.21it/s]


wrote cached tensors -> /home/pchati/MHCPrime/dataset_cache.pt (16.9 MB)


Ranking score columns: 100%|██████████| 1/1 [00:00<00:00, 53.20it/s]


,seq,allele,label,mhcprime,mhcprime_rank
0,AEQIKHILA,B4101,1,21.25000,99.973770
1,MEDETQKEL,B4101,1,-11.81250,99.250877
2,AEIEPEAQA,B4101,1,73.50000,99.998695
3,KESEVLLQA,B4101,1,19.50000,99.970520
4,AEDPSQLP,B4101,1,7.25000,99.914841
5,AEFKEAFQL,B4101,1,-3.43750,99.596138
6,KEVEETATA,B4101,1,34.00000,99.988327
7,REYWNEQSA,B4101,1,20.12500,99.971794
8,KEDSQPPTP,B4101,1,3.84375,99.866982
9,REVQGFESA,B4101,1,24.12500,99.978516


Cache exists after inference: False
CPU times: user 1min 6s, sys: 1.9 s, total: 1min 8s
Wall time: 7.3 s
